In [47]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [2]:
ML_PATH = "../data/movielens-1m/movielens_1m_ratings.csv"

In [3]:
df = pd.read_csv(ML_PATH)

In [6]:
pivoted = df.pivot_table(index='user', columns='item', values='rating', fill_value=0)

In [10]:
pivoted.loc[[3, 4], 6]

user
3    0.0
4    0.0
Name: 6, dtype: float64

In [ ]:
class MFDataLoader(Dataset):
    def __init__(self, file_path):
        self.target_column = 'rating'
        self._columns = ['user', 'item', self.target_column]
        self.data = pd.read_csv(file_path)[self._columns]
        self.n_users = self.data['user'].nunique()
        self.n_items = self.data['item'].nunique()

        unique_user_ids = self.data['user'].unique()
        unique_item_ids = self.data['item'].unique()
        self.user_id_map = {old_id: new_id for new_id, old_id in enumerate(sorted(unique_user_ids))}
        self.item_id_map = {old_id: new_id for new_id, old_id in enumerate(sorted(unique_item_ids))}
        # We preprocess df to use standardized ids, so that we don´t need to worry about it
        self.data['user'] = self.data['user'].map(self.user_id_map)
        self.data['item'] = self.data['item'].map(self.item_id_map)



    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        user = int(row['user'])
        item = int(row['item'])
        label = float(row[self.target_column])
        return (user, item), label

In [100]:
df.groupby("item").agg({"user": list}).loc[2]

user    [10, 13, 18, 23, 27, 40, 44, 48, 53, 60, 62, 7...
Name: 2, dtype: object

In [210]:
from sklearn.neighbors import NearestNeighbors

class UserKNN:
    def __init__(self, mf_data_loader, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.mf_data_loader = mf_data_loader
        self._fit()

    def _fit(self):
        df = self.mf_data_loader.data
        user_item_matrix = df.pivot_table(index='user', columns='item', values='rating', fill_value=0)
        self.user_item_matrix = user_item_matrix
        self.model = NearestNeighbors(n_neighbors=self.n_neighbors, metric='cosine')
        self.model.fit(user_item_matrix.values)
        self.item_rated2user = df.groupby("item")["user"].apply(list)
       
        self.users_biases = df.groupby("user")["rating"].mean()

    def get_neighbors(self, user_id):
        user_vector = self.user_item_matrix.iloc[user_id].values.reshape(1, -1)
        distances, indices = self.model.kneighbors(user_vector, n_neighbors=self.n_neighbors + 1)
        # drop self: Sklearns knn module always return the point itself as a part of its 
        #neighborhodd
        distances, indices = distances[:, 1:], indices[:, 1:]
        sims = 1 - distances.squeeze()
        neighbor_ids = self.user_item_matrix.index[indices[0]].tolist()
        return neighbor_ids, sims
    
    def predict(self, user_id, item_id):
        assert 0 <= user_id < self.mf_data_loader.n_users, f"user_id must be valid and within range [0, {self.mf_data_loader.n_users})"
        assert 0 <= item_id < self.mf_data_loader.n_items, f"item_id must be valid and within range [0, {self.mf_data_loader.n_items})"

        user_bias = self.users_biases.loc[user_id]

        neighbor_ids, distances = self.get_neighbors(user_id=user_id)
        
        rated_item = self.item_rated2user.loc[item_id]
        filtered_neighborhood = [(neighbor, dist) for neighbor, dist in zip(neighbor_ids, distances) if neighbor in rated_item]
        
        if(len(filtered_neighborhood) > 0):
            filtered_neighbors, filtered_sims = zip(*filtered_neighborhood)
            filtered_neighbors = list(filtered_neighbors)
            filtered_sims = np.array(filtered_sims)
            neighbors_biases = np.array(self.users_biases[filtered_neighbors].tolist())
            neighborhood_ratings_of_item_id = np.array(self.user_item_matrix.loc[filtered_neighbors, item_id].tolist())
            neighbors_ratings_sans_bias = neighborhood_ratings_of_item_id - neighbors_biases
            weighted_rating = np.dot(neighbors_ratings_sans_bias, filtered_sims).item()
            return user_bias + (weighted_rating / np.sum(filtered_sims).item())
        else:
            return user_bias

    
    


In [211]:

uknn = UserKNN(loader)

In [209]:
np.array(uknn.users_biases[[0,2,3]].tolist()) - np.array([4,3,4])

array([0.18867925, 0.90196078, 0.19047619])

In [143]:
neighbor_ids, distances = uknn.get_neighbors(user_id=1)
        
rated_item = uknn.item_rated2user.loc[1]
filtered_neighborhood = [(neighbor, dist) for neighbor, dist in zip(neighbor_ids, distances) if neighbor in rated_item]


In [180]:
filtered_neighborhood

[(3107, np.float64(0.39589399303623163)),
 (4600, np.float64(0.362874983310018))]

In [181]:

if(len(filtered_neighborhood) > 0):
    filtered_neighbors, filtered_sims = zip(*filtered_neighborhood)
    filtered_neighbors = list(filtered_neighbors)
    filtered_sims = np.array(filtered_sims)

In [206]:
uknn.users_biases[filtered_neighbors].tolist()

[3.68801652892562, 3.62015503875969]

In [182]:
filtered_sims

array([0.39589399, 0.36287498])

In [185]:
neighborhood_ratings_of_item_id = np.array(uknn.user_item_matrix.loc[filtered_neighbors, 1].to_list())

In [195]:
filtered_sims

array([0.39589399, 0.36287498])

In [190]:
np.sum(filtered_sims).item()

0.7587689763462496

In [198]:
uknn.users_biases.loc[1]

np.float64(3.7131782945736433)

In [199]:
neighborhood_ratings_of_item_id

array([2., 4.])

In [200]:
filtered_sims

array([0.39589399, 0.36287498])

In [201]:
np.dot(neighborhood_ratings_of_item_id, filtered_sims).item()

2.243287919312535

In [197]:
np.dot(neighborhood_ratings_of_item_id, filtered_sims).item() / np.sum(filtered_sims).item()

2.9564834478536373

In [147]:
filtered_sims

(np.float64(0.39589399303623163), np.float64(0.362874983310018))

In [212]:
uknn.predict(1, 1)

np.float64(3.014099409546858)

In [116]:
user_vector = uknn.user_item_matrix.iloc[1].values.reshape(1, -1)


In [117]:
user_vector

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 3706))

In [129]:
1 - uknn.model.kneighbors(user_vector)[0].squeeze()

array([1.        , 0.39589399, 0.37348298, 0.36477744, 0.36287498])

In [ ]:
distances, indices = uknn.model.kneighbors(user_vector)

In [194]:
uknn.predict(1, 1)

np.float64(6.66966174242728)